# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [ ]:
%%bash
# Define the environment name
QI_NAME="qiime2-amplicon-2024.10"

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi

### Run the ASV pipeline

In [ ]:

./asv_pipeline.sh --config asv.conf

### Run General Statistics

In [ ]:
mamba activate asv-py
THREADS="$(nproc)"

seqkit stat -a -T -o ../spark_refactored_output/stats/fastq_stats.tsv -j ${THREADS} ../fastq_combined/*.fastq.gz
seqkit stat -a -T -o ../spark_refactored_output/stats/fastp_fastqs.tsv -j ${THREADS} ../spark_refactored_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../spark_refactored_output/stats/filtered_fastqs.tsv -j ${THREADS} ../spark_refactored_output/filtered/*.fasta
seqkit stat -a -T -o ../spark_refactored_output/stats/concat_fastas.tsv -j ${THREADS} ../spark_refactored_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier

In [ ]:
conda activate qiime2-amplicon-2024.10
mkdir -p ../spark_refactored_output/taxonomy
awk '/^>/ {print; next} {print toupper($0)}' ../spark_refactored_output/ASVs/ASVs.fasta > ../spark_refactored_output/ASVs/ASVs.upper.fasta
python qiime_vs_classifier.py \
  --input-fasta ../spark_refactored_output/ASVs/ASVs.upper.fasta \
  --ref-taxonomy ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output ../spark_refactored_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Mitomaster, decontamination, mitoDB

In [ ]:
mamba activate asv-py

rm -rf ../spark_refactored_output/mito
mkdir -p ../spark_refactored_output/mito/mitomap
rm -rf ../spark_refactored_output/ASVs/chunks

seqkit split -s 10 -O ../spark_refactored_output/ASVs/chunks ../spark_refactored_output/ASVs/ASVs_filtered.fasta

python ./mitomaster.py \
       --data-dir ../spark_refactored_output/ASVs/chunks/ \
       --output-file ../spark_refactored_output/mito/mitomap/mitomaster_output.tsv

blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/mito_ncbi \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/mito_ncbi.blast6.tsv

blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/ssu_pipeline_contaminants \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv

python ./mito_checker.py \
       --mitomaster-file ../spark_refactored_output/mito/mitomap/mitomaster_output.tsv \
       --mito-blast ../spark_refactored_output/mito/mitomap/mito_ncbi.blast6.tsv \
       --silva-tax ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
       --biof-file ../spark_refactored_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv \
       --output-dir ../spark_refactored_output/mito/mitomap/ --overwrite

### Filter ASV count tables

In [ ]:
mkdir -p ../spark_refactored_output/mito/ASVs
python filter_nontarget.py \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASV_filtered.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/nontarget.master.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASV_target.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    0.005

### Build Sankey Diagram

In [ ]:
mamba deactivate
mkdir -p ../spark_refactored_output/metadata
mkdir -p ../spark_refactored_output/mito/metadata

python sankey_builder.py \
    --data-dir ../ \
    --sub-dir spark_refactored_output \
    --metadata ../ref_db/combined_metadata.tsv \
    --make-labeled --make-unlabeled

### Plot Metadata

In [ ]:
python plot_metadata.py \
    --data-dir ../ \
    --sub-dir spark_refactored_output \
    --metadata ../ref_db/spark_metadata.tsv \
    --taxonomy ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --asv-micro ../spark_refactored_output/ASVs/ASV_target.micro.tsv \
    --asv-mito ../spark_refactored_output/mito/ASVs/ASV_target.mito.tsv \
    --make-micro --make-mito

python outlier_checker.py \
    --data-dir ../ \
    --metadata spark_refactored_output/metadata/metadata_updated_micro.tsv \
    --output spark_refactored_output/metadata \
    --asv spark_refactored_output/ASVs/ASV_target.micro.tsv \
    --group-cols type_group

python collectors_curve.py \
    --counts ../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --meta ../ref_db/spark_metadata.tsv \
    --sample-id-col sample \
    --group-col type_group \
    --out_prefix ../spark_refactored_output/metadata/collectors_curve \
    --permutations 999 --seed 42 \
    --group-colors "Oral Rinse=#6A3D9A,BAL=#0072B2,Lung Brush=#009E73" \
    --group-order "Oral Rinse,BAL,Lung Brush"

### Plot Upset

In [ ]:

python plot_upset.py \
    --data-dir ../ \
    --subdir spark_refactored_output \
    --domain both \
    --taxonomy-path ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --formats svg,pdf --do-composite-oral-lung

python venn_bubbles.py \
    --data-dir ../ \
    --subdir spark_refactored_output \
    --asv-meta ../spark_refactored_output/metadata/ASV_meta_micro.tsv \
    --presence ../spark_refactored_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --type-order "Oral Rinse,BAL,Lung Brush" --formats svg,pdfs

### Run Alpha and Beta Diversity

In [ ]:

python calc_div.py \
    --micro-table ../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --mito-table ../spark_refactored_output/mito/ASVs/ASV_final.mito.tsv \
    --outdir ../spark_refactored_output/diversity \
    --mito-outdir ../spark_refactored_output/mito/diversity 

python plot_diversity.py \
    --metadata ../spark_refactored_output/metadata/metadata_updated_micro.tsv \
    --master ../spark_refactored_output/metadata/master_table_micro.tsv \
    --alpha ../spark_refactored_output/diversity/shannon.tsv \
    --bray ../spark_refactored_output/diversity/bray.tsv \
    --jacc ../spark_refactored_output/diversity/jaccard.tsv \
    --outliers-all ../spark_refactored_output/metadata/outliers_type_group.tsv \
    --type-order "Oral Rinse,BAL,Lung Brush" \
    --mito-alpha ../spark_refactored_output/mito/diversity/shannon.mito.tsv \
    --mito-bray ../spark_refactored_output/mito/diversity/bray.mito.tsv \
    --mito-jacc ../spark_refactored_output/mito/diversity/jaccard.mito.tsv \
    --outdir ../spark_refactored_output/diversity/ \
    --mito-outdir ../spark_refactored_output/mito/diversity/ \
    --exclude-types "Skin Brush,Scope Flush"

### Run indicspecies (R)

In [ ]:
Rscript run_indicspecies.R \
    --asv ../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --meta ../spark_refactored_output/metadata/metadata_updated_micro.tsv \
    --sample-col sample \
    --group-cols status,type_group \
    --outdir ../spark_refactored_output

### Plot indicspecies Results

In [ ]:
python plot_indicspecies.py \
    --type-results ../spark_refactored_output/indicspecies/type_group_indicator_species_results.tsv \
    --status-results ../spark_refactored_output/indicspecies/status_indicator_species_results.tsv \
    --venn ../spark_refactored_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --taxonomy ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --outdir ../spark_refactored_output/indicspecies/ \
    --type-index  "1=BAL,2=Lung Brush,3=Oral Rinse,4=BAL+Lung Brush,5=BAL+Oral Rinse,6=Lung Brush+Oral Rinse,7=Oral Rinse+BAL+Lung Brush" \
    --status-index "1=Cancer,2=Non-Cancer,3=Cancer+Non-Cancer" \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3" \
    --status-markers "Non-Cancer=D,Cancer=X,Cancer+Non-Cancer=^,not_indicator=o"
    

### Plot Clustermaps

In [ ]:
python plot_clustermaps.py \
    --asv-meta ../spark_refactored_output/metadata/ASV_meta_micro.tsv \
    --metadata ../spark_refactored_output/metadata/metadata_updated_micro.tsv \
    --isa ../spark_refactored_output/indicspecies/Type_status_ISA_results.tsv \
    --outdir ../spark_refactored_output/diversity \
    --type-order "Oral Rinse,BAL,Lung Brush" \
    --exclude-types "Skin Brush,Scope Flush" \
    --mito-asv ../spark_refactored_output/mito/ASVs/ASV_final.mito.tsv \
    --mito-outdir ../spark_refactored_output/mito/diversity \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3"
    

### Run SPIEC-EASI (R)

In [ ]:
Rscript run_spieceasi.R \
    --counts=../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --outdir=../spark_refactored_output/spieceasi \
    --force-graphs TRUE

### Graph Network

In [ ]:
python graph_network.py \
    --data-dir ../spark_refactored_output/ \
    --outdir ../spark_refactored_output/spieceasi \
    --graph-pos-all ../spark_refactored_output/spieceasi/spieceasi_network_pos_all.graphml \
    --graph-pos-sub ../spark_refactored_output/spieceasi/spieceasi_network_pos_thr.graphml \
    --node-features ../spark_refactored_output/spieceasi/spieceasi_node_features.csv \
    --asv-counts ../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --taxonomy ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --venn ../spark_refactored_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --type-summary ../spark_refactored_output/indicspecies/type_group_indicator_species_summary.tsv \
    --status-summary ../spark_refactored_output/indicspecies/status_indicator_species_summary.tsv
    